# 01 — Data Exploration
Load and inspect real hardware jumping data from `.mat` files.

**Outputs:** Nothing saved to disk — this notebook is read-only exploration.

**Run before:** `02_training.ipynb`

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from hopper import load_jumping_data, print_dataset_summary
from hopper.evaluation import plot_data_distribution

# Reproducibility
np.random.seed(42)

## 2. Configuration

In [ ]:
DATA_DIR = "/Users/cassandrahe/MIT Dropbox/Cassandra He/jumping_data"

DOWNSAMPLE_FACTOR = 5   # 500 Hz -> 100 Hz
CLIP_START_SEC    = 3.0 # Remove initial transient
CLIP_END_SEC      = 13.0

FEATURE_NAMES = [
    'pos_x', 'pos_y', 'pos_z',
    'roll', 'pitch', 'yaw',
    'thrust',
    'tau_x', 'tau_y', 'tau_z',
    'f1', 'f2', 'f3', 'f4'
]

OUTPUT_NAMES = [
    'Δpos_x', 'Δpos_y', 'Δpos_z',
    'Δroll',  'Δpitch', 'Δyaw'
]

## 3. Load Data

In [ ]:
print("Loading jumping data...")
all_data = load_jumping_data(
    data_dir=DATA_DIR,
    downsample_factor=DOWNSAMPLE_FACTOR,
    clip_start_sec=CLIP_START_SEC,
    clip_end_sec=CLIP_END_SEC,
)

X = all_data.X  # [N, 14]: [pos(3), eul(3), thrust(1), torque(3), signals(4)]
Y = all_data.Y  # [N, 6]:  [delta_pos(3), delta_eul(3)]

print(f"\nLoaded: {X.shape[0]} samples")
print(f"Input dim:  {X.shape[1]}")
print(f"Output dim: {Y.shape[1]}")

## 4. Summary Statistics

In [ ]:
print_dataset_summary(X, Y)

## 5. Feature & Target Distributions

In [ ]:
plot_data_distribution(X, Y, FEATURE_NAMES, OUTPUT_NAMES)

## 6. Time-Series Inspection
Plot raw trajectories to sanity-check the hardware data.

In [ ]:
dt = 1.0 / 100.0  # 100 Hz after downsampling
t  = np.arange(len(all_data.pos_ds)) * dt

fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)

# Position
for i, label in enumerate(['x', 'y', 'z']):
    axes[0].plot(t, all_data.pos_ds[:, i], label=label)
axes[0].set_ylabel('Position [m]')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_title('Raw Hardware Trajectories')

# Euler angles
for i, label in enumerate(['roll', 'pitch', 'yaw']):
    axes[1].plot(t, np.degrees(all_data.eul_ds[:, i]), label=label)
axes[1].set_ylabel('Angle [deg]')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Wing signals
for i in range(4):
    axes[2].plot(t, all_data.signals_ds[:, i], label=f'f{i+1}', alpha=0.7)
axes[2].set_ylabel('Wing Signal')
axes[2].set_xlabel('Time [s]')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Delta Distribution Check
Verify the output targets (deltas) are well-behaved — large outliers here
will hurt training.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6))
axes = axes.flatten()

for i, name in enumerate(OUTPUT_NAMES):
    axes[i].hist(Y[:, i], bins='auto', alpha=0.7, edgecolor='black', color='steelblue')
    axes[i].set_title(name)
    axes[i].set_xlabel('Delta value')
    axes[i].axvline(0, color='r', linestyle='--', alpha=0.5)
    axes[i].grid(True, alpha=0.3)

    # Flag if std is suspiciously large
    std = Y[:, i].std()
    axes[i].set_title(f'{name}  (std={std:.2e})')

plt.suptitle('Output Delta Distributions')
plt.tight_layout()
plt.show()

# Flag any outliers
for i, name in enumerate(OUTPUT_NAMES):
    outliers = np.sum(np.abs(Y[:, i]) > 5 * Y[:, i].std())
    if outliers > 0:
        print(f"[WARN] {name}: {outliers} outliers beyond 5σ")